# Step 8: the block and the stack

This notebook goes with step 8 of `vak`. It builds one block by hand, then the whole GPT at a small size, and counts
its weights. `uv run vak model` builds the real one.

Before this: step 7 runs many heads, and adds position. After this: step 9 trains the GPT.

| Section | Idea | Used in |
|---|---|---|
| 1 | The block: attention, then the MLP, each added to the residual stream | `vak model` |
| 2 | The stack: the whole GPT, its weights, and its untrained guess | `vak model` |

## 1. The block

A **block** is one layer of the GPT. Attention lets the tokens read each other, then the **MLP** works on each
token alone: it widens the vector 4 times, applies GELU, and narrows it back. Each part reads a normalized copy of
x (**LayerNorm**) and adds its change to x. So x, the **residual stream**, runs through every block, and each block
only adds to it. Here is one block on 3 tokens of 8 numbers, done by hand next to the block itself.

In [1]:
import torch
from medha.model import GPT, Block, Config

config = Config(vocab_size=7, context=8, width=8, heads=2, layers=2)
torch.manual_seed(0)
block = Block(config)
x = torch.randn(1, 3, 8)
after_attention = x + block.attention(block.norm1(x))
after_mlp = after_attention + block.mlp(block.norm2(after_attention))
print("x, token 0:          ", [round(number, 2) for number in x[0, 0].tolist()])
print("by hand, token 0:    ", [round(number, 2) for number in after_mlp[0, 0].tolist()])
print("the block, token 0:  ", [round(number, 2) for number in block(x)[0, 0].tolist()])
print("shape in:", tuple(x.shape), " shape out:", tuple(block(x).shape))

x, token 0:           [-1.02, -1.53, -1.22, 0.87, -0.38, 0.1, -0.32, 1.85]
by hand, token 0:     [-1.22, -1.51, -1.32, 1.56, -0.01, 0.42, -0.25, 2.41]
the block, token 0:   [-1.22, -1.51, -1.32, 1.56, -0.01, 0.42, -0.25, 2.41]
shape in: (1, 3, 8)  shape out: (1, 3, 8)


**What you see**

The block's output for token 0, `[-1.22, -1.51, -1.32, 1.56, -0.01, 0.42, -0.25, 2.41]`, equals the hand version:
attention adds its change, then the MLP adds its own. The output stays close to x, because each part only adds to
it. The shape is the same in and out, `(1, 3, 8)`, so blocks can be stacked as deep as we like.

Used in: `uv run vak model` runs 4 such blocks, of 4 heads each, with vectors of 128 numbers.

## 2. The stack

The **GPT** stacks the blocks: the Embedding gives each token its vector, the blocks run in turn, a last LayerNorm
tidies the result, and the **head** turns each vector into one logit per vocabulary token. The head reuses the
token table. Count its weights by hand, and run it, untrained, on "the cat sat on the mat".

In [2]:
torch.manual_seed(0)
gpt = GPT(config)
tables = 7 * 8 + 8 * 8  # the token table, which the head reuses, and the position table
attention_weights = 8 * 24 + 24 + 8 * 8 + 8  # q, k and v from one Linear, then the output Linear
mlp_weights = 8 * 32 + 32 + 32 * 8 + 8
norms = 2 * (8 + 8)
by_hand = tables + 2 * (attention_weights + mlp_weights + norms) + (8 + 8)
counted = 0
for weights in gpt.parameters():
    counted += weights.numel()
print("weights by hand:", by_hand, " counted:", counted)

words = ["<|endoftext|>", "the", "cat", "sat", "on", "mat", "dog"]
story = torch.tensor([[0, 1, 2, 3, 4, 1, 5, 0]])  # <|endoftext|> the cat sat on the mat <|endoftext|>
logits = gpt(story)
print("logits:", tuple(logits.shape))
chances = torch.softmax(logits[0, -1], dim=-1)
print("chances after the last token:", [round(chance, 3) for chance in chances.tolist()])
print("an even guess over 7 tokens:", round(1 / 7, 3))

weights by hand: 1880  counted: 1880
logits: (1, 8, 7)
chances after the last token: [0.159, 0.146, 0.153, 0.15, 0.117, 0.134, 0.14]
an even guess over 7 tokens: 0.143


**What you see**

By hand and counted, the small GPT has `1880` weights. The logits have the shape `(1, 8, 7)`: one row of 7
logits at each of the 8 places, each a guess at the next token. Untrained, the chances lie between `0.117` and
`0.159`, near an even guess of `0.143`: the weights start small, so no token is favoured much. Training changes
that.

Used in: `uv run vak model` builds the real GPT, logs the shape after each part, and counts its 1,065,728
weights part by part.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Block | x + attention(norm(x)), then x + MLP(norm(x)): the same shape in and out | the GPT stacks 4 of them |
| MLP | widen each vector 4 times, GELU, narrow it back: most of the weights | every block |
| The stack | Embedding, the blocks, a last norm, then the head: one logit per vocabulary token at each place | step 9 trains its 1,065,728 weights |
| Untrained | small weights give every token about the same chance: a loss near ln(vocabulary size) | step 9's loss starts there |